# Notebook 4 – Paper Trading Walkthrough
> Bar-by-bar simulation using PaperExecutor with stops and risk controls.
>
> **Educational only. Paper trading results may not reflect live performance.**

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from data.fetcher import fetch_yfinance
from src.strategies import MomentumStrategy
from src.executor import PaperExecutor
from src.risk_manager import fixed_pct_size, compute_stops, PortfolioRiskState, is_stop_hit
from src.utils import setup_logging

setup_logging('INFO')
TICKER = 'QQQ'
CAPITAL = 100_000

In [ ]:
df = fetch_yfinance(TICKER, start='2022-01-01', end='2024-01-01')
strategy  = MomentumStrategy(allow_short=False)
executor  = PaperExecutor(initial_capital=CAPITAL)
risk_state = PortfolioRiskState(peak_equity=CAPITAL, current_equity=CAPITAL)
open_stops = {}
equity_history = []

In [ ]:
# Bar-by-bar loop
WARMUP = 60

for i in range(WARMUP, len(df)):
    bar = df.iloc[:i + 1]
    price = float(bar['Close'].iloc[-1])
    executor.set_price(TICKER, price)

    signals_df = strategy.generate_signals(bar)
    equity = executor.get_equity()
    risk_state.update(equity)
    equity_history.append(equity)

    positions = executor.get_positions()
    held = positions.get(TICKER)
    held_qty = held.qty if held else 0

    # The signal uses this bar's close, so fill on the next bar.
    # A drawdown halt flattens; it does not leave the open book on.
    if len(signals_df) < 2:
        continue
    if risk_state.check_halt():
        if held_qty != 0:
            executor.close_position(TICKER)
            open_stops.pop(TICKER, None)
        continue
    decision = signals_df.iloc[-2]
    position = int(decision['Position'])

    # Check stop/take-profit levels
    if TICKER in open_stops:
        sl, tp, direction = open_stops[TICKER]
        hit, reason = is_stop_hit(price, direction, sl, tp)
        if hit:
            print(f'  {df.index[i].date()} – {reason} triggered @ {price:.2f}')
            executor.close_position(TICKER)
            del open_stops[TICKER]
            continue

    # Enter long
    if position == 1 and held_qty <= 0:
        qty = fixed_pct_size(equity, price, risk_pct=0.05)
        if qty > 0:
            executor.submit_order(TICKER, 'buy', qty)
            stops = compute_stops(price, direction=1, stop_loss_pct=0.03, take_profit_pct=0.06)
            open_stops[TICKER] = (stops.stop_loss, stops.take_profit, 1)
    # Exit flat
    elif position == 0 and held_qty > 0:
        executor.close_position(TICKER)
        open_stops.pop(TICKER, None)

final_equity = executor.get_equity()
print(f'\nFinal equity : ${final_equity:,.2f}')
print(f'Return       : {(final_equity/CAPITAL - 1):.2%}')

In [ ]:
eq_series = pd.Series(equity_history, index=df.index[WARMUP:])

plt.figure(figsize=(12, 4))
plt.plot(eq_series, label='Paper Portfolio', color='steelblue', lw=1.5)
plt.axhline(CAPITAL, linestyle='--', color='gray', alpha=0.6, label='Initial Capital')
plt.legend()
plt.title(f'Paper Trading Result – Momentum on {TICKER}')
plt.ylabel('Portfolio Value ($)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Trade log
log = executor.get_trade_log()
print(f'Total filled orders: {len(log)}')
log[['timestamp', 'symbol', 'side', 'qty', 'filled_price']].head(20)